# Feature funnel

Measures completeness and emptiness for every master-table variable, classifies each from the standardized type catalog, and shows counts by source table and completeness threshold.

**Project step:** §4.2 step 1.0.

**Inputs:** `data/master_listing.parquet`, `data/master_columns.csv`.

**Outputs:** `outputs/02_phase_01_feature_funnel{suffix}.csv`.

**Run after:** 01_master_table_check.

**Runtime:** ~3 min.


In [1]:
# ---- Data ----
INPUT = "data/master_listing.parquet"   # relative to the repo root
OUTPUT_DIR = "outputs"                  # every CSV / XLSX / PNG goes here

# ---- Filters (None = no filter) ----
CITY = None             # "chicago", "dfw", "dc", or a list of them       -> column `market`
NEIGHBORHOOD = None     # submarket/list -> column `market_building__submarket`
START_DATE = None       # "YYYY-MM-DD": keep listings with DATE_COLUMN >= START_DATE
END_DATE = None         # "YYYY-MM-DD": keep listings with DATE_COLUMN <= END_DATE
DATE_COLUMN = "market_unit__enter_market"


In [2]:
from pathlib import Path

import pandas as pd

BLUE, GREEN, YELLOW, RED, BOLD, RESET = "\033[94m", "\033[92m", "\033[93m", "\033[91m", "\033[1m", "\033[0m"


def step(msg):
    print(f"\n{BOLD}{BLUE}▶ {msg}{RESET}")


def done(msg):
    print(f"{GREEN}  ✓ {msg}{RESET}")


def note(msg):
    print(f"{YELLOW}  • {msg}{RESET}")


ROOT = Path.cwd()
while not (ROOT / "data" / "master_listing.parquet").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
MASTER = ROOT / INPUT
CATALOG = ROOT / "data" / "master_columns.csv"
OUTPUTS = ROOT / OUTPUT_DIR
OUTPUTS.mkdir(parents=True, exist_ok=True)
NN = 2
FILTER_COLUMNS = ["market", "market_building__submarket", DATE_COLUMN]


def _as_list(value):
    """Normalize one filter value or a list of values."""
    if value is None:
        return None
    return [value] if isinstance(value, str) else list(value)


def suffix():
    """Build the shared output suffix from the active filters."""
    parts = []
    if CITY is not None:
        parts.append("+".join(map(str, _as_list(CITY))))
    if NEIGHBORHOOD is not None:
        parts.append("+".join(map(str, _as_list(NEIGHBORHOOD))))
    if START_DATE is not None or END_DATE is not None:
        parts.append(f"{START_DATE or 'start'}_to_{END_DATE or 'end'}")
    return "" if not parts else "__" + "__".join(parts)


def out(name, ext="csv"):
    """Build an output path with this notebook's number and active filters."""
    return OUTPUTS / f"{NN:02d}_phase_01_{name}{suffix()}.{ext}"


def upstream(number, name, ext="csv"):
    """Build an upstream notebook's output path with the active filters."""
    return OUTPUTS / f"{number:02d}_phase_01_{name}{suffix()}.{ext}"


def need(path, notebook):
    """Raise a clear instruction when a required upstream output is missing."""
    if not Path(path).exists():
        raise FileNotFoundError(f"Run {notebook} first; missing {path}")


def load(columns):
    """Read requested master columns and apply the configured listing filters."""
    requested = list(dict.fromkeys(columns))
    read_columns = list(dict.fromkeys(requested + FILTER_COLUMNS))
    frame = pd.read_parquet(MASTER, columns=read_columns)
    valid_values = {
        column: sorted(frame[column].dropna().astype(str).unique())
        for column in ("market", "market_building__submarket")
    }
    for column, value, label in (
        ("market", CITY, "city"),
        ("market_building__submarket", NEIGHBORHOOD, "neighborhood"),
    ):
        if value is not None:
            wanted = list(map(str, _as_list(value)))
            valid = valid_values[column]
            unknown = sorted(set(wanted) - set(valid))
            if unknown:
                raise ValueError(f"Unknown {label}(s) {unknown}. Valid values: {valid}")
            before = len(frame)
            frame = frame[frame[column].astype(str).isin(wanted)]
            note(f"{label}: {before:,} → {len(frame):,} rows")
    if START_DATE is not None:
        before = len(frame)
        frame = frame[frame[DATE_COLUMN] >= pd.Timestamp(START_DATE)]
        note(f"start date: {before:,} → {len(frame):,} rows")
    if END_DATE is not None:
        before = len(frame)
        frame = frame[frame[DATE_COLUMN] <= pd.Timestamp(END_DATE)]
        note(f"end date: {before:,} → {len(frame):,} rows")
    return frame[requested]


In [3]:
step(f"Reading {MASTER.name} and {CATALOG.name}")
catalog = pd.read_csv(CATALOG)
variables = catalog["column"].tolist()
frame = load(variables)
n_rows = len(frame)
funnel = pd.DataFrame({"variable": variables})
done(f"{n_rows:,} rows x {len(funnel)} variables")

step("Extracting the source table from each variable name (split on '__')")
# The 4 join keys (unit_id, property_id, floor_plan_id, market) carry no prefix;
# they come from market.unit.
funnel["source_table"] = [v.split("__")[0] if "__" in v else "market_unit" for v in funnel["variable"]]
done(f"{funnel['source_table'].nunique()} source tables")
for table, n in funnel["source_table"].value_counts().items():
    note(f"{table:32s}{n:>4} variables")

step("Computing completeness and emptiness (% of filtered rows non-null / null)")
funnel["completeness_pct"] = (frame.notna().mean().to_numpy() * 100).round(2)
funnel["emptiness_pct"] = (100 - funnel["completeness_pct"]).round(2)
done(f"{(funnel['completeness_pct'] == 100).sum()} variables are 100% complete")
if (funnel["completeness_pct"] == 0).any():
    print(f"{RED}  ✗ fully empty: {funnel.loc[funnel['completeness_pct'] == 0, 'variable'].tolist()}{RESET}")

step("Classifying variables from the standardized type catalog")
type_map = {"numeric": "number", "string": "string", "boolean": "boolean", "datetime": "date"}
funnel = funnel.merge(catalog[["column", "type_group"]],
                      left_on="variable", right_on="column", how="left")
funnel["type"] = funnel["type_group"].map(type_map).fillna("other")
for kind, g in funnel.groupby("type"):
    note(f"{kind:8s}{len(g):>4} variables, mean completeness {g['completeness_pct'].mean():6.2f}%")
if (funnel["type"] == "other").any():
    unmapped = sorted(funnel.loc[funnel["type"] == "other", "type_group"].unique())
    print(f"{RED}  ✗ unmapped type groups: {unmapped}{RESET}")

step("Funnel: variables kept at each completeness threshold")
for label, kept in [("all variables", len(funnel))] + [
        (f">= {t}% complete", (funnel["completeness_pct"] >= t).sum()) for t in (50, 80, 95, 100)]:
    color = GREEN if kept / len(funnel) >= 0.8 else YELLOW if kept / len(funnel) >= 0.5 else RED
    print(f"  {label:18s}{color}{'█' * round(40 * kept / len(funnel)):40s} {kept:>4}{RESET}")

OUT = out("feature_funnel")
step(f"Writing {OUT.relative_to(ROOT)}")
funnel = funnel[["variable", "source_table", "completeness_pct", "emptiness_pct", "type"]]
funnel.to_csv(OUT, index=False)
done(f"{len(funnel)} rows x {funnel.shape[1]} columns")
funnel


▶ Reading master_listing.parquet and master_columns.csv


  ✓ 967,661 rows x 378 variables

▶ Extracting the source table from each variable name (split on '__')
  ✓ 11 source tables
  • market_building                  270 variables
  • truth_building_params             23 variables
  • market_price_observation          20 variables
  • market_unit                       18 variables
  • pms_property                      15 variables
  • truth_lease_event                 13 variables
  • market_concession_event            9 variables
  • truth_concession                   5 variables
  • market_leasing_activity_daily      2 variables
  • truth_floorplan_market_rent        2 variables
  • market_floor_plan                  1 variables

▶ Computing completeness and emptiness (% of filtered rows non-null / null)


  ✓ 71 variables are 100% complete
  ✗ fully empty: ['market_building__overall_quality_score', 'market_building__bedroom_quality_score', 'market_building__kitchen_quality_score', 'market_building__bathroom_quality_score', 'market_building__dining_room_quality_score', 'market_building__common_areas_quality_score', 'market_building__fitness_center_quality_score', 'market_building__laundry_room_quality_score', 'market_building__living_room_quality_score', 'market_building__main_entrance_quality_score', 'market_building__stairs_hallway_quality_score', 'market_building__swimming_pool_quality_score', 'market_building__walk_in_closet_quality_score']

▶ Classifying variables from the standardized type catalog
  • boolean  223 variables, mean completeness  81.09%
  • date      11 variables, mean completeness  83.31%
  • number   108 variables, mean completeness  66.03%
  • string    36 variables, mean completeness  78.00%

▶ Funnel: variables kept at each completeness threshold
  all variables 

,variable,source_table,completeness_pct,emptiness_pct,type
0,unit_id,market_unit,100.00,0.00,string
1,property_id,market_unit,100.00,0.00,string
2,floor_plan_id,market_unit,100.00,0.00,string
3,market,market_unit,100.00,0.00,string
4,market_unit__unit_name,market_unit,89.22,10.78,string
...,...,...,...,...,...
373,pms_property__comps_disabled,pms_property,5.39,94.61,boolean
374,pms_property__pricing_disabled,pms_property,5.39,94.61,boolean
375,pms_property__fetch_pms_pricing_disabled,pms_property,5.39,94.61,boolean
376,pms_property__num_available_vacant_units,pms_property,5.39,94.61,number
